[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/00_entorno/00_proyecto_entorno.ipynb)

# Proyecto 00 · Levanta el stack, verifica su salud y crea tu primer dashboard

| | |
|---|---|
| **Nivel** | 🟢 Básico |
| **Duración estimada** | ≈ 1 h 45 min |
| **Servicios de Compose** | `core` + `metrics` (≈ 0,5 GB de RAM). Sin ellos, el proyecto se evalúa en **modo degradado** (solo las comprobaciones sin red) |
| **Prerrequisitos** | Lección `00_entorno` |
| **Conexión** | Es la base de **SupportOps**: todos los proyectos posteriores reutilizan `check_stack()`, `backends` y el dashboard que crees aquí |

## Contexto de negocio

Eres ingeniero/a de plataforma en **SupportOps**, una tienda online de hardware que está a punto de poner un agente de soporte en producción.
Antes de instrumentar nada, tu jefa de plataforma te pide un **entorno de observabilidad reproducible** para el equipo: que cualquiera pueda levantarlo con un comando,
que un *script de verificación* confirme que de verdad funciona de extremo a extremo (no solo que los contenedores estén «Up») y que exista un **primer dashboard como código**.

## Datos

No hay dataset: la telemetría de prueba la generas tú (una traza, un log y una métrica «hola mundo»). Cadena de resiliencia: **stack local → modo degradado** (las funciones puras
se comprueban sin red con el *validador offline* incluido; las comprobaciones online se omiten y se marcan «no evaluable»).

## Entregables y criterios de evaluación (rúbrica)

| # | Entregable | Criterio medido por la solución de referencia | Umbral | Modo |
|---|---|---|---|---|
| R1 | Los 4 servicios `core` listos + Prometheus | `check_stack(["core","metrics"])` sin servicios requeridos faltantes | 5/5 | online |
| R2 | Salud de datasources | `service_health_report()` devuelve OK para `tempo`, `loki`, `prometheus` | 3/3 (Pyroscope puede fallar) | online |
| R3 | Payload de traza válido | El validador offline acepta tu traza: 3 spans, jerarquía raíz→hijos, ids hex correctos | 0 errores del validador | **offline** |
| R4 | Telemetría de extremo a extremo | Tu `trace_id` se encuentra en Tempo (3 spans) y en Loki (1 log con `trace_id` como metadato) | 100 % de las 3 señales encontradas en ≤ 90 s | online |
| R5 | Dashboard como código | JSON con ≥ 3 paneles de ≥ 2 tipos distintos y datasources `prometheus`+`loki` referenciados por uid | 0 errores del validador | **offline** |
| R6 | Dashboard publicado | `GET /api/dashboards/uid/<tu uid>` existe y tiene los mismos paneles | igual nº de paneles | online |

> 📏 **Calibración:** estos umbrales salen de lo que mide la solución de referencia (al final) en el stack real: 5/5 servicios, 3/3 datasources, traza de 3 spans visible en ≈ 15-30 s, log con `trace_id`, dashboard con 3 paneles.
> No dependen de la velocidad de tu máquina (solo del plazo de 90 s).

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz]"], check=True)
print("Curso en:", ROOT)

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, ...
from supportops import viz, sample_data, schema, backends
viz.apply_style()
pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 80)
SEED = 42
FAST_DEV_RUN = True   # True: pocos tickets (iteración rápida). False: tamaños «completos» del texto

import json, secrets, time, urllib.request, shutil, subprocess
STACK = check_stack(["core", "metrics"])

## Paso 0 · (fuera del notebook) Levanta el stack

En una terminal, desde `agent-observability-course/stack/`:

```bash
docker compose --profile core --profile metrics up -d
```

Espera ≈ 20 s y vuelve a ejecutar la celda anterior hasta que `check_stack` diga *modo completo*. (Si no puedes usar Docker, sigue: los pasos 2, 4 y 6 se evalúan offline.)

## Paso 1 · Informe de salud de servicios

**TODO 1.** Implementa `service_health_report()` que devuelva un `DataFrame` con una fila por servicio (`collector, tempo, loki, grafana, prometheus`), con columnas `servicio`, `estado` (`up`, `starting`, `down`) y `url`.
Reutiliza lo que ya sabes: `supportops.stack.services()` te da las URLs de salud, y `check_stack(...).up` te dice cuáles responden.

<details>
<summary>🪜 Pista 1</summary>

Cada servicio tiene una URL de salud en `supportops.stack.services()[nombre]["url"]`. `check_stack(require, verbose=False)` devuelve un objeto con `.up` (dict `nombre → bool`) y `.starting` (lista de nombres que responden pero aún arrancan).

</details>

<details>
<summary>🪜 Pista 2</summary>

Construye una lista de diccionarios `{"servicio": n, "estado": ..., "url": ...}` iterando sobre `["collector","tempo","loki","grafana","prometheus"]` y pásala a `pd.DataFrame`. Estado: `"up"` si `st.up[n]`, `"starting"` si `n in st.starting`, si no `"down"`.

</details>

In [ ]:
from supportops.stack import services

def service_health_report() -> pd.DataFrame:
    # TODO 1: devuelve un DataFrame con columnas servicio, estado, url (una fila por servicio de core + prometheus)
    raise NotImplementedError("TODO 1")

try:
    display(service_health_report())
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 2 · Construye los payloads OTLP/JSON

**TODO 2.** Completa `build_hello(service, n_requests)` para que devuelva `(trace_id, {"/v1/traces": ..., "/v1/logs": ..., "/v1/metrics": ...})` con:

- **Traza** de **3 spans**: raíz `invoke_agent hola` (kind 1 = INTERNAL), hijo `chat mock-small` (kind 3 = CLIENT) y hermano `execute_tool get_order` (kind 1) — ambos hijos de la raíz, sin solaparse, dentro de la raíz.
- **Log** `INFO` con cuerpo `"hola mundo"`, con el `traceId` y el `spanId` de la raíz y el atributo `ticket.id`.
- **Métrica** *counter* acumulado (`sum`, `aggregationTemporality` 2, `isMonotonic` true) llamada `hello_requests` con valor `n_requests`.

Reglas OTLP/JSON recordadas: ids en **hex** (32/16), tiempos en nanosegundos como **string**, atributos como `{"key","value":{"stringValue":...}}`.

<details>
<summary>🪜 Pista 1</summary>

Parte de la función `hello_payloads()` de la lección: tenías 2 spans; añade un tercero con otro `spanId`, mismo `parentSpanId` (el de la raíz) y tiempos que no se solapen con el primer hijo, p. ej. `now-150ms → now-20ms`.

</details>

<details>
<summary>🪜 Pista 2</summary>

El `traceId` es **el mismo** en los 3 spans y en el log (`secrets.token_hex(16)`); cada `spanId` es distinto (`secrets.token_hex(8)`). El log usa `traceId` y `spanId` de la raíz. La métrica no lleva ids.

</details>

In [ ]:
def attrs(**kv):
    def val(v):
        return {"intValue": str(v)} if isinstance(v, int) and not isinstance(v, bool) else {"boolValue": v} if isinstance(v, bool) else {"stringValue": str(v)}
    return [{"key": k, "value": val(v)} for k, v in kv.items()]

def build_hello(service: str = "proyecto-00", n_requests: int = 5):
    # TODO 2: construye trace_id y los 3 payloads (traza de 3 spans, log correlacionado, métrica counter)
    raise NotImplementedError("TODO 2")

try:
    TRACE_ID, PAYLOADS = build_hello()
    print("trace_id:", TRACE_ID)
except NotImplementedError as e:
    TRACE_ID, PAYLOADS = None, None
    print("Pendiente:", e)

## Paso 3 · Validador offline (te lo damos hecho)

Para comprobar tus payloads **sin red**, usa este validador. Devuelve la lista de errores (vacía = correcto). Léelo: es una buena revisión del modelo de datos de OTLP.

In [ ]:
import re
HEX = lambda s, n: isinstance(s, str) and re.fullmatch(rf"[0-9a-f]{{{n}}}", s) is not None

def validate_trace_payload(p: dict) -> list[str]:
    errs = []
    try:
        spans = [s for rs in p["resourceSpans"] for ss in rs["scopeSpans"] for s in ss["spans"]]
    except (KeyError, TypeError):
        return ["estructura resourceSpans → scopeSpans → spans incorrecta"]
    if len(spans) != 3: errs.append(f"se esperaban 3 spans, hay {len(spans)}")
    tids = {s.get("traceId") for s in spans}
    if len(tids) != 1 or not all(HEX(t, 32) for t in tids): errs.append("traceId distinto entre spans o no es hex de 32")
    ids = [s.get("spanId") for s in spans]
    if not all(HEX(i, 16) for i in ids) or len(set(ids)) != len(ids): errs.append("spanId no es hex de 16 o está repetido")
    roots = [s for s in spans if not s.get("parentSpanId")]
    if len(roots) != 1: errs.append(f"debe haber exactamente 1 raíz, hay {len(roots)}")
    else:
        r = roots[0]; rs_, re_ = int(r["startTimeUnixNano"]), int(r["endTimeUnixNano"])
        for s in spans:
            if s is r: continue
            if s.get("parentSpanId") != r["spanId"]: errs.append(f"{s['name']}: el padre no es la raíz")
            a, b = int(s["startTimeUnixNano"]), int(s["endTimeUnixNano"])
            if not (rs_ <= a <= b <= re_): errs.append(f"{s['name']}: no está contenido en la raíz")
        kids = sorted((s for s in spans if s is not r), key=lambda s: int(s["startTimeUnixNano"]))
        if len(kids) == 2 and int(kids[0]["endTimeUnixNano"]) > int(kids[1]["startTimeUnixNano"]): errs.append("los dos hijos se solapan")
    names = {s["name"] for s in spans}
    for need in ("invoke_agent hola", "chat mock-small", "execute_tool get_order"):
        if need not in names: errs.append(f"falta el span «{need}»")
    for s in spans:
        if not isinstance(s.get("startTimeUnixNano"), str): errs.append("los tiempos deben ser strings")
    return errs

def validate_log_payload(p: dict, trace_id: str) -> list[str]:
    try:
        recs = [r for rl in p["resourceLogs"] for sl in rl["scopeLogs"] for r in sl["logRecords"]]
    except (KeyError, TypeError):
        return ["estructura resourceLogs → scopeLogs → logRecords incorrecta"]
    errs = []
    if len(recs) != 1: return [f"se esperaba 1 log, hay {len(recs)}"]
    r = recs[0]
    if r.get("severityNumber") != 9 or r.get("severityText") != "INFO": errs.append("severidad debe ser INFO (9)")
    if r.get("body", {}).get("stringValue") != "hola mundo": errs.append("cuerpo distinto de «hola mundo»")
    if r.get("traceId") != trace_id or not HEX(r.get("spanId"), 16): errs.append("el log no está correlacionado (traceId/spanId)")
    if "ticket.id" not in {a["key"] for a in r.get("attributes", [])}: errs.append("falta el atributo ticket.id")
    return errs

def validate_metric_payload(p: dict, n: int) -> list[str]:
    try:
        ms = [m for rm in p["resourceMetrics"] for sm in rm["scopeMetrics"] for m in sm["metrics"]]
    except (KeyError, TypeError):
        return ["estructura resourceMetrics → scopeMetrics → metrics incorrecta"]
    if len(ms) != 1 or ms[0]["name"] != "hello_requests": return ["debe haber una métrica llamada hello_requests"]
    s = ms[0].get("sum", {}); dp = (s.get("dataPoints") or [{}])[0]
    errs = []
    if s.get("aggregationTemporality") != 2 or s.get("isMonotonic") is not True: errs.append("debe ser sum acumulada (temporality 2) y monotónica")
    if dp.get("asInt") != str(n): errs.append(f"el valor debe ser {n} (asInt como string)")
    return errs

def validate_all(trace_id, payloads, n=5):
    if payloads is None: return {"traza": ["sin implementar"], "log": ["sin implementar"], "métrica": ["sin implementar"]}
    return {"traza": validate_trace_payload(payloads["/v1/traces"]), "log": validate_log_payload(payloads["/v1/logs"], trace_id),
            "métrica": validate_metric_payload(payloads["/v1/metrics"], n)}

print(validate_all(TRACE_ID, PAYLOADS))

## Paso 4 · Envía y verifica de extremo a extremo

**TODO 3.** Implementa `send_and_verify(trace_id, payloads, timeout=90)`:

1. `POST` de cada payload al Collector (`http://localhost:4318` + la ruta).
2. *Polling* con plazo: busca la traza en Tempo (`backends.tempo_trace`) y comprueba que tiene **3 spans**; busca el log en Loki con LogQL `{service_name="proyecto-00"} | trace_id="<id>"` y la métrica en Prometheus (`hello_requests_total`).
3. Devuelve un `dict` `{"traza": bool, "log": bool, "metrica": bool, "segundos": float}`.

Si el Collector no está disponible, devuelve `None` (modo degradado).

<details>
<summary>🪜 Pista 1</summary>

El `polling` de la lección (`wait_for`) ya hace casi todo. Cuenta los spans de Tempo con `sum(len(ss['spans']) for b in doc['batches'] for ss in b['scopeSpans'])`.

</details>

<details>
<summary>🪜 Pista 2</summary>

`backends.loki_query(logql, minutes=15)` devuelve una lista (vacía si no hay nada). `backends.prom_query('hello_requests_total')` devuelve una lista de series. Tempo tarda 15-30 s: usa un `timeout` de 90 s y una pausa de 2 s.

</details>

In [ ]:
def wait_for(fn, timeout=90, every=2.0):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            r = fn()
            if r: return r
        except backends.BackendError:
            pass
        time.sleep(every)
    return None

def post(path, payload, base="http://localhost:4318"):
    req = urllib.request.Request(base + path, data=json.dumps(payload).encode(), headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=5) as r:
        return r.status

def send_and_verify(trace_id, payloads, timeout=90):
    # TODO 3: enviar, esperar y verificar en Tempo, Loki y Prometheus
    raise NotImplementedError("TODO 3")

CHECK_STACK = check_stack(["core", "metrics"], verbose=False)
VERIFY = None
if PAYLOADS and CHECK_STACK.up.get("collector") and not any(validate_all(TRACE_ID, PAYLOADS).values()):
    try:
        VERIFY = send_and_verify(TRACE_ID, PAYLOADS)
        print(VERIFY)
    except NotImplementedError as e:
        print("Pendiente:", e)
else:
    print("Sin stack, sin payloads o con errores de validación: salto el envío.")

## Paso 5 · Tu primer dashboard, **como código**

**TODO 4.** Completa `build_dashboard()` para devolver el JSON de un dashboard con **uid** `hola-mundo-NN` (elige tú el sufijo) y título «Hola mundo», con al menos 3 paneles de al menos 2 tipos:

- un `stat` con la consulta PromQL `hello_requests_total` (datasource Prometheus, uid `prometheus`),
- un `timeseries` con `sum(rate(otelcol_receiver_accepted_spans[1m]))` (Prometheus),
- un panel de **logs** (`type: "logs"`) con la consulta LogQL `{service_name="proyecto-00"}` (datasource Loki, uid `loki`).

Cada panel necesita `id` único, `title`, `type`, `gridPos` (`x,y,w,h`; el lienzo mide 24 de ancho), `datasource` (`{"type": ..., "uid": ...}`) y `targets` (con `refId` y `expr`).

<details>
<summary>🪜 Pista 1</summary>

Mira `stack/grafana/dashboards/00-stack-overview.json`: es un dashboard real del repo con paneles `stat` y `timeseries`. Copia su estructura (`panels`, `gridPos`, `targets`).

</details>

<details>
<summary>🪜 Pista 2</summary>

Estructura mínima de un panel: `{"id": 1, "type": "stat", "title": "...", "datasource": {"type": "prometheus", "uid": "prometheus"}, "gridPos": {"x": 0, "y": 0, "w": 8, "h": 6}, "targets": [{"refId": "A", "expr": "..."}]}`. Para Loki el `type` del datasource es `"loki"`. Dashboard: `{"uid": ..., "title": ..., "schemaVersion": 39, "panels": [...]}`.

</details>

In [ ]:
DASH_UID = "hola-mundo-00"

def build_dashboard() -> dict:
    # TODO 4: devuelve el dict del dashboard (uid DASH_UID, título «Hola mundo», ≥ 3 paneles de ≥ 2 tipos)
    raise NotImplementedError("TODO 4")

def validate_dashboard(d: dict) -> list[str]:
    errs = []
    if d.get("uid") != DASH_UID: errs.append(f"uid debe ser {DASH_UID}")
    if not d.get("title"): errs.append("falta title")
    panels = d.get("panels", [])
    if len(panels) < 3: errs.append(f"se piden ≥ 3 paneles, hay {len(panels)}")
    if len({p.get('type') for p in panels}) < 2: errs.append("se piden ≥ 2 tipos de panel distintos")
    ids = [p.get("id") for p in panels]
    if len(set(ids)) != len(ids) or None in ids: errs.append("ids de panel repetidos o ausentes")
    uids = set()
    for p in panels:
        gp = p.get("gridPos", {})
        if not all(k in gp for k in "xywh") or gp.get("x", 0) + gp.get("w", 0) > 24: errs.append(f"gridPos incorrecto en «{p.get('title')}»")
        ds = p.get("datasource", {})
        uids.add(ds.get("uid"))
        if not p.get("targets") or not all("refId" in t and ("expr" in t) for t in p["targets"]): errs.append(f"targets incompletos en «{p.get('title')}»")
    if not {"prometheus", "loki"} <= uids: errs.append("deben usarse los datasources prometheus y loki (por uid)")
    if not any(p.get("type") == "logs" for p in panels): errs.append("falta un panel de tipo logs")
    return errs

try:
    DASH = build_dashboard()
    print("errores del validador:", validate_dashboard(DASH))
except NotImplementedError as e:
    DASH = None
    print("Pendiente:", e)

## Paso 6 · Publica el dashboard y compruébalo

Dos formas de hacer un dashboard «como código»: (a) **provisionado** (copiar el JSON a `stack/grafana/dashboards/` y Grafana lo carga en ≤ 10 s) o (b) vía **API** de Grafana.
Usaremos la API para poder verificarlo desde aquí.

**TODO 5.** Implementa `publish_dashboard(d)`: `POST /api/dashboards/db` con `{"dashboard": d, "overwrite": True}` (usa `backends.grafana_post`) y después `GET /api/dashboards/uid/<uid>` (usa `backends.grafana_api`) para devolver el número de paneles guardado. Devuelve `None` si Grafana no está disponible.

<details>
<summary>🪜 Pista 1</summary>

`backends.grafana_post("/api/dashboards/db", {"dashboard": d, "overwrite": True})` devuelve un dict con `status: 'success'`. Para releerlo: `backends.grafana_api(f"/api/dashboards/uid/{d['uid']}")['dashboard']['panels']`.

</details>

In [ ]:
def publish_dashboard(d: dict):
    # TODO 5: publicar vía API de Grafana y devolver len(paneles) releídos
    raise NotImplementedError("TODO 5")

PUBLISHED = None
if DASH and not validate_dashboard(DASH) and check_stack("grafana", verbose=False).up.get("grafana"):
    try:
        PUBLISHED = publish_dashboard(DASH)
        print("paneles guardados en Grafana:", PUBLISHED, "→ ábrelo en http://localhost:3000/d/" + DASH_UID)
    except NotImplementedError as e:
        print("Pendiente:", e)
else:
    print("Salto la publicación (sin Grafana, sin dashboard o con errores).")

## Paso 7 · Autoevaluación con la rúbrica

Ejecuta la celda: calcula cada criterio. Lo que no se puede evaluar (porque no hay stack) aparece como **no evaluable**, no como fallo.

In [ ]:
def run_rubric() -> pd.DataFrame:
    rows = []
    st = check_stack(["core", "metrics"], verbose=False)
    def add(r, desc, ok, detail=""):
        rows.append({"criterio": r, "descripción": desc, "resultado": "—  no evaluable" if ok is None else ("✅ OK" if ok else "❌ FALLA"), "detalle": detail})
    # R1, R2 (online)
    try:
        rep = service_health_report()
        add("R1", "5/5 servicios listos", None if st.degraded and not st.up.get("collector") else bool((rep["estado"] == "up").all()), str(rep["estado"].value_counts().to_dict()))
    except NotImplementedError:
        add("R1", "5/5 servicios listos", False, "TODO 1 sin implementar")
    if st.up.get("grafana"):
        h = {u: backends.grafana_api(f"/api/datasources/uid/{u}/health").get("status") == "OK" for u in ["tempo", "loki", "prometheus"]}
        add("R2", "datasources tempo/loki/prometheus OK", all(h.values()), str(h))
    else:
        add("R2", "datasources tempo/loki/prometheus OK", None)
    # R3 (offline)
    errs = validate_all(TRACE_ID, PAYLOADS)
    add("R3", "payloads OTLP válidos (offline)", PAYLOADS is not None and not any(errs.values()), json.dumps(errs, ensure_ascii=False)[:160])
    # R4 (online)
    add("R4", "traza+log+métrica encontradas ≤ 90 s", None if VERIFY is None else all(VERIFY[k] for k in ("traza", "log", "metrica")) and VERIFY["segundos"] <= 90,
        str(VERIFY))
    # R5 (offline)
    derr = validate_dashboard(DASH) if DASH else ["sin implementar"]
    add("R5", "dashboard válido (offline)", not derr, "; ".join(derr)[:160])
    # R6 (online)
    add("R6", "dashboard publicado con sus paneles", None if PUBLISHED is None and not st.up.get("grafana") else (DASH is not None and PUBLISHED == len(DASH["panels"])), str(PUBLISHED))
    return pd.DataFrame(rows).set_index("criterio")

display(run_rubric())

---
## Retos extra (nivel experto)

1. **Provisionado en lugar de API:** guarda tu dashboard en `stack/grafana/dashboards/01-hola-mundo.json`, espera ≤ 10 s y comprueba con `grafana_api` que aparece en la carpeta «Curso observabilidad de agentes». ¿Qué pasa si cambias el `uid` sin borrar el volumen? (pista: mira la tabla de pitfalls de la lección).
2. **Canario permanente:** convierte `send_and_verify` en un script `canary.py` que un `cron`/GitHub Action pueda ejecutar y que termine con código de salida ≠ 0 si falta alguna señal. ¿Qué *timeout* eliges y por qué?
3. **Perfil propio:** añade a `docker-compose.yml` (en una copia) un servicio opcional bajo el perfil `debug` con `otel/opentelemetry-collector-contrib` configurado con el exporter `debug`, y compara qué ve cada Collector.
4. **Presupuesto de memoria:** añade `mem_limit` a cada servicio (calcula el valor a partir de `docker stats` bajo carga con `generate_tickets(500, start_s="live")` enviado a mano) y razona qué pasa cuando Tempo llega al límite.

## Reflexión (conecta con producción y con el hilo de SupportOps)

- ¿Por qué verificar **de extremo a extremo** (traza canario) es mejor que mirar `docker compose ps`? ¿Qué fallos solo detecta el canario?
- Tu equipo tendrá 5 agentes. ¿Un Collector por agente o uno compartido (*gateway*)? ¿Qué cambia en fallos, costes y seguridad? (lo retomamos en el módulo 05)
- ¿Qué parte de este entorno pondrías bajo control de versiones y cuál *no* (datos, contraseñas)? ¿Qué harías con la contraseña `admin/admin` en un entorno compartido?
- En el módulo 01 abrimos el agente SupportOps y empezamos a generar tráfico: ¿qué dos preguntas de negocio esperas poder contestar mirando Grafana al terminar el curso?

---
# ⛔ SPOILER — intenta resolverlo primero

Debajo está la **solución de referencia**, completa y ejecutable. Si te atascas, usa antes las pistas 🪜 de cada paso.

In [ ]:
# ===================== SOLUCIÓN DE REFERENCIA =====================
# TODO 1
def service_health_report() -> pd.DataFrame:
    names = ["collector", "tempo", "loki", "grafana", "prometheus"]
    st = check_stack(names, verbose=False)
    svc = services()
    return pd.DataFrame([{"servicio": n, "estado": "up" if st.up.get(n) else ("starting" if n in st.starting else "down"), "url": svc[n]["url"]}
                         for n in names])

# TODO 2
def build_hello(service: str = "proyecto-00", n_requests: int = 5):
    now = time.time_ns()
    trace_id, root, c1, c2 = secrets.token_hex(16), secrets.token_hex(8), secrets.token_hex(8), secrets.token_hex(8)
    resource, scope = {"attributes": attrs(**{"service.name": service})}, {"name": "proyecto.00", "version": "0"}
    span = lambda sid, name, kind, a, b, parent=None, **at: {**{"traceId": trace_id, "spanId": sid, "name": name, "kind": kind,
        "startTimeUnixNano": str(a), "endTimeUnixNano": str(b), "attributes": attrs(**at), "status": {"code": 1}},
        **({"parentSpanId": parent} if parent else {})}
    traces = {"resourceSpans": [{"resource": resource, "scopeSpans": [{"scope": scope, "spans": [
        span(root, "invoke_agent hola", 1, now - 400_000_000, now, **{"gen_ai.operation.name": "invoke_agent"}),
        span(c1, "chat mock-small", 3, now - 350_000_000, now - 150_000_000, root, **{"gen_ai.operation.name": "chat"}),
        span(c2, "execute_tool get_order", 1, now - 120_000_000, now - 30_000_000, root, **{"gen_ai.tool.name": "get_order"})]}]}]}
    logs = {"resourceLogs": [{"resource": resource, "scopeLogs": [{"scope": scope, "logRecords": [
        {"timeUnixNano": str(now), "severityNumber": 9, "severityText": "INFO", "body": {"stringValue": "hola mundo"},
         "traceId": trace_id, "spanId": root, "attributes": attrs(**{"ticket.id": "T-1"})}]}]}]}
    metrics = {"resourceMetrics": [{"resource": resource, "scopeMetrics": [{"scope": scope, "metrics": [
        {"name": "hello_requests", "unit": "{request}", "sum": {"aggregationTemporality": 2, "isMonotonic": True,
         "dataPoints": [{"startTimeUnixNano": str(now - 60_000_000_000), "timeUnixNano": str(now), "asInt": str(n_requests)}]}}]}]}]}
    return trace_id, {"/v1/traces": traces, "/v1/logs": logs, "/v1/metrics": metrics}

# TODO 3
def send_and_verify(trace_id, payloads, timeout=90):
    t0 = time.time()
    for path, payload in payloads.items():
        post(path, payload)
    n_spans = lambda doc: sum(len(ss["spans"]) for b in doc["batches"] for ss in b["scopeSpans"])
    tr = wait_for(lambda: (lambda d: d if n_spans(d) == 3 else None)(backends.tempo_trace(trace_id)), timeout)
    lg = wait_for(lambda: backends.loki_query(f'{{service_name="proyecto-00"}} | trace_id="{trace_id}"', minutes=15), timeout)
    mt = wait_for(lambda: backends.prom_query("hello_requests_total"), timeout) if check_stack("prometheus", verbose=False).up.get("prometheus") else None
    return {"traza": bool(tr), "log": bool(lg), "metrica": bool(mt), "segundos": round(time.time() - t0, 1)}

# TODO 4
def build_dashboard() -> dict:
    P, L = {"type": "prometheus", "uid": "prometheus"}, {"type": "loki", "uid": "loki"}
    return {"uid": DASH_UID, "title": "Hola mundo", "schemaVersion": 39, "editable": True, "time": {"from": "now-30m", "to": "now"}, "panels": [
        {"id": 1, "type": "stat", "title": "hello_requests", "datasource": P, "gridPos": {"x": 0, "y": 0, "w": 6, "h": 6},
         "targets": [{"refId": "A", "expr": "hello_requests_total", "instant": True}]},
        {"id": 2, "type": "timeseries", "title": "Spans aceptados por el Collector (por segundo)", "datasource": P, "gridPos": {"x": 6, "y": 0, "w": 18, "h": 6},
         "targets": [{"refId": "A", "expr": "sum(rate(otelcol_receiver_accepted_spans[1m]))"}]},
        {"id": 3, "type": "logs", "title": "Logs de proyecto-00", "datasource": L, "gridPos": {"x": 0, "y": 6, "w": 24, "h": 8},
         "targets": [{"refId": "A", "expr": '{service_name="proyecto-00"}'}]}]}

# TODO 5
def publish_dashboard(d: dict):
    r = backends.grafana_post("/api/dashboards/db", {"dashboard": d, "overwrite": True})
    assert r.get("status") == "success", r
    return len(backends.grafana_api(f"/api/dashboards/uid/{d['uid']}")["dashboard"]["panels"])

# --- ejecutar la solución de arriba a abajo ---
print(service_health_report())
TRACE_ID, PAYLOADS = build_hello()
print("validación offline:", validate_all(TRACE_ID, PAYLOADS))
CHECK_STACK = check_stack(["core", "metrics"], verbose=False)
if CHECK_STACK.up.get("collector"):
    VERIFY = send_and_verify(TRACE_ID, PAYLOADS); print("extremo a extremo:", VERIFY)
DASH = build_dashboard(); print("dashboard:", validate_dashboard(DASH))
if check_stack("grafana", verbose=False).up.get("grafana"):
    PUBLISHED = publish_dashboard(DASH); print("paneles publicados:", PUBLISHED)
display(run_rubric())